# Projeto de Visão Computacional - YOLO
## Hugo Mariano - RM560688
### Fase 6 - PBL | FarmTech Solutions

Este notebook documenta o desenvolvimento completo do sistema de visão computacional com YOLO customizado, conforme o planejamento do projeto e as tasks das fases 3 e 4.

## Objetivo
- Detectar e classificar dois objetos distintos usando YOLO customizado
- Comparar desempenho entre diferentes configurações de treinamento
- Testar o modelo em imagens separadas e analisar resultados


## 1. Importação

Importamos todas as bibliotecas necessárias para manipulação de dados, visualização e avaliação dos modelos.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
import tensorflow as tf
import sklearn
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import os
import glob
import random
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report

## 2. Configuração dos Caminhos e Parâmetros

Configuramos os caminhos dos dados, parâmetros de treino e as classes do projeto. Utilizamos o arquivo `data.yaml` já existente no repositório para configurar o dataset e as classes.

In [2]:
# Caminhos principais
train_dir = '../dataset/train/images'
val_dir = '../dataset/val/images'
test_dir = '../dataset/test/images'
labels_dir = '../dataset/train/labels'
models_dir = '../models/'
results_dir = '../results/yolo_custom/'
data_yaml = '../config/data.yaml'

# Hiperparâmetros
epochs_1 = 30
epochs_2 = 60
batch_size = 16
learning_rate = 0.001
imgsz = 640

# Criação dos diretórios de saída, se não existirem
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Verificamos se os diretórios de dados existem
for path in [train_dir, val_dir, test_dir]:
    if not os.path.exists(path):
        print(f"AVISO: O caminho {path} não existe!")

# Definimos as classes do projeto (apenas para visualização, as classes reais estão no data.yaml)
classes = ['copo', 'headphones']

## 3. Visualização de Amostras do Dataset

Visualizamos exemplos das imagens para garantir que o dataset está correto.

In [ ]:
def show_images_from_folder(folder, n=4):
    imgs = glob.glob(os.path.join(folder, '*.jpg')) + glob.glob(os.path.join(folder, '*.png'))
    if not imgs:
        print(f"Nenhuma imagem encontrada em {folder}")
        return
    random.shuffle(imgs)
    plt.figure(figsize=(15, 4))
    for i, img_path in enumerate(imgs[:n]):
        img = Image.open(img_path)
        plt.subplot(1, n, i+1)
        plt.imshow(img)
        plt.title(f"{os.path.basename(img_path)}\n{img.size}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f"Total de imagens: {len(imgs)}")

print('Amostras de treino:')
show_images_from_folder(train_dir)
print('Amostras de validação:')
show_images_from_folder(val_dir)
print('Amostras de teste:')
show_images_from_folder(test_dir)

## 4. Treinamento do Modelo YOLO Customizado

Treinamos dois modelos YOLO com diferentes números de épocas para comparar desempenho. Também garantimos que o ambiente está pronto para o treinamento e que a configuração do dataset está correta.

In [5]:

# Treinamento com 30 épocas
print("Iniciando treinamento com 30 épocas...")
model = YOLO('yolov8n.pt')  # Modelo base
results_30 = model.train(
    data=data_yaml, 
    epochs=epochs_1, 
    batch=batch_size, 
    imgsz=imgsz, 
    project=models_dir, 
    name='yolo_30ep',
    patience=10,  # Early stopping
    verbose=True
)
model.save(os.path.join(models_dir, 'model_30epochs.pt'))
print(f"Modelo de 30 épocas salvo em {os.path.join(models_dir, 'model_30epochs.pt')}")

# Treinamento com 60 épocas
print("Iniciando treinamento com 60 épocas...")
results_60 = model.train(
    data=data_yaml, 
    epochs=epochs_2, 
    batch=batch_size, 
    imgsz=imgsz, 
    project=models_dir, 
    name='yolo_60ep',
    patience=10,  # Early stopping
    verbose=True
)
model.save(os.path.join(models_dir, 'model_60epochs.pt'))
print(f"Modelo de 60 épocas salvo em {os.path.join(models_dir, 'model_60epochs.pt')}")

Iniciando treinamento com 30 épocas...
New https://pypi.org/project/ultralytics/8.3.121 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.120  Python-3.12.3 torch-2.7.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=../config/data.yaml, epochs=30, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=../models/, name=yolo_30ep2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=Fal

100%|██████████| 755k/755k [00:00<00:00, 9.87MB/s]

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  1     37248  ultralytics.nn.modules.block.C2f             [192, 64, 1]                  
 16                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  1    123648  ultralytics.nn.modules.block.C2f             [192, 128, 1]                 
 19                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  1    493056  ultralytics.nn.modules.block.C2f             [384, 256, 1]                 
 22        [15, 18, 21]  1    751702  ultralytics.nn.modules.head.Detect           [2, [

train: Scanning C:\Users\gugue\OneDrive\Área de Trabalho\Fiap\Fase 6\trabalho_fase6\trabalho_fase6_fiap_farmtech\projeto_visao_computacional\dataset\train\labels... 80 images, 0 backgrounds, 0 corrupt: 100%|██████████| 80/80 [00:00<00:00, 302.83it/s]

...

      12/60         0G      1.005      1.228      1.552         36        640: 100%|██████████| 5/5 [00:39<00:00,  7.92s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.55s/it]

                   all         10         11      0.818      0.818      0.842      0.484
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 2, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



12 epochs completed in 0.147 hours.
Optimizer stripped from ..\models\yolo_60ep\weights\last.pt, 6.2MB
Optimizer stripped from ..\models\yolo_60ep\weights\best.pt, 6.2MB

Validating ..\models\yolo_60ep\weights\best.pt...
Ultralytics 8.3.120  Python-3.12.3 torch-2.7.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]


                   all         10         11      0.994      0.818      0.871       0.66
                  copo         10         11      0.994      0.818      0.871       0.66
Speed: 3.1ms preprocess, 137.2ms inference, 0.0ms loss, 45.3ms postprocess per image
Results saved to ..\models\yolo_60ep
Modelo de 60 épocas salvo em ../models/model_60epochs.pt


## 5. Avaliação dos Modelos

Avaliamos o desempenho dos modelos treinados usando métricas padrão e comparamos os resultados.

In [8]:
def display_metrics(metrics, title):
    print(f"\n=== {title} ===")
    # Caso metrics seja um dicionário (versões recentes do Ultralytics)
    if isinstance(metrics, dict) and 'metrics' in metrics:
        m = metrics['metrics']
        print(f"mAP@0.5: {m['map50']:.4f}")
        print(f"mAP@0.5:0.95: {m['map']:.4f}")
        print(f"Precisão: {m['precision']:.4f}")
        print(f"Recall: {m['recall']:.4f}")
        f1 = (2 * m['precision'] * m['recall']) / (m['precision'] + m['recall'] + 1e-6)
        print(f"F1-Score: {f1:.4f}")
    # Caso metrics.box seja um dicionário (algumas versões intermediárias)
    elif hasattr(metrics, 'box') and isinstance(metrics.box, dict):
        b = metrics.box
        print(f"mAP@0.5: {b['map50']:.4f}")
        print(f"mAP@0.5:0.95: {b['map']:.4f}")
        print(f"Precisão: {b['precision']:.4f}")
        print(f"Recall: {b['recall']:.4f}")
        f1 = (2 * b['precision'] * b['recall']) / (b['precision'] + b['recall'] + 1e-6)
        print(f"F1-Score: {f1:.4f}")
    else:
        print("Estrutura de métricas não reconhecida. Veja o print abaixo para detalhes.")
        print(metrics)

print("Avaliando modelo de 30 épocas...")
model_30 = YOLO(os.path.join(models_dir, 'model_30epochs.pt'))
metrics_30 = model_30.val(data=data_yaml, batch=batch_size, imgsz=imgsz)
display_metrics(metrics_30, "Métricas do modelo (30 épocas)")

print("Avaliando modelo de 60 épocas...")
model_60 = YOLO(os.path.join(models_dir, 'model_60epochs.pt'))
metrics_60 = model_60.val(data=data_yaml, batch=batch_size, imgsz=imgsz)
display_metrics(metrics_60, "Métricas do modelo (60 épocas)")

# Comparação visual
comparison = {
    'mAP@0.5': [metrics_30.box.map50, metrics_60.box.map50],
    'mAP@0.5:0.95': [metrics_30.box.map, metrics_60.box.map],
    'Precisão': [metrics_30.box.precision, metrics_60.box.precision],
    'Recall': [metrics_30.box.recall, metrics_60.box.recall]
}

plt.figure(figsize=(10, 6))
df = pd.DataFrame(comparison, index=['30 épocas', '60 épocas'])
ax = df.plot(kind='bar', rot=0, width=0.7)
plt.title('Comparação de Métricas entre os Modelos', fontsize=14)
plt.ylabel('Valor', fontsize=12)
plt.xlabel('Modelo', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=10)
plt.tight_layout()
plt.show()

Avaliando modelo de 30 épocas...
Ultralytics 8.3.120  Python-3.12.3 torch-2.7.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 195.254.5 MB/s, size: 29.0 KB)


val: Scanning C:\Users\gugue\OneDrive\Área de Trabalho\Fiap\Fase 6\trabalho_fase6\trabalho_fase6_fiap_farmtech\projeto_visao_computacional\dataset\val\labels.cache... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<?, ?it/s]
c:\Users\gugue\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.63s/it]


                   all         10         11      0.852      0.727      0.799      0.531
                  copo         10         11      0.852      0.727      0.799      0.531
Speed: 2.0ms preprocess, 117.6ms inference, 0.0ms loss, 35.7ms postprocess per image
Results saved to runs\detect\val3

=== Métricas do modelo (30 épocas) ===
Estrutura de métricas não reconhecida. Veja o print abaixo para detalhes.
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017B4D6AD880>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,   

val: Scanning C:\Users\gugue\OneDrive\Área de Trabalho\Fiap\Fase 6\trabalho_fase6\trabalho_fase6_fiap_farmtech\projeto_visao_computacional\dataset\val\labels.cache... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<?, ?it/s]
c:\Users\gugue\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.40s/it]


                   all         10         11      0.994      0.818      0.871       0.66
                  copo         10         11      0.994      0.818      0.871       0.66
Speed: 2.9ms preprocess, 92.0ms inference, 0.0ms loss, 37.9ms postprocess per image
Results saved to runs\detect\val4

=== Métricas do modelo (60 épocas) ===
Estrutura de métricas não reconhecida. Veja o print abaixo para detalhes.
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017B44984B00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    

AttributeError: 'Metric' object has no attribute 'precision'. See valid attributes below.

    Class for computing evaluation metrics for YOLOv8 model.

    Attributes:
        p (list): Precision for each class. Shape: (nc,).
        r (list): Recall for each class. Shape: (nc,).
        f1 (list): F1 score for each class. Shape: (nc,).
        all_ap (list): AP scores for all classes and all IoU thresholds. Shape: (nc, 10).
        ap_class_index (list): Index of class for each AP score. Shape: (nc,).
        nc (int): Number of classes.

    Methods:
        ap50(): AP at IoU threshold of 0.5 for all classes. Returns: List of AP scores. Shape: (nc,) or [].
        ap(): AP at IoU thresholds from 0.5 to 0.95 for all classes. Returns: List of AP scores. Shape: (nc,) or [].
        mp(): Mean precision of all classes. Returns: Float.
        mr(): Mean recall of all classes. Returns: Float.
        map50(): Mean AP at IoU threshold of 0.5 for all classes. Returns: Float.
        map75(

## 6. Visualização das Previsões em Imagens de Validação

Visualizamos as previsões dos modelos para analisar qualitativamente os resultados.

In [ ]:
def show_predictions_comparison(img_path):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    img = Image.open(img_path)
    axes[0].imshow(img)
    axes[0].set_title('Imagem Original', fontsize=12)
    axes[0].axis('off')
    results_30 = model_30(img_path, conf=0.25)
    axes[1].imshow(results_30[0].plot())
    axes[1].set_title('Previsão - Modelo 30 épocas', fontsize=12)
    axes[1].axis('off')
    results_60 = model_60(img_path, conf=0.25)
    axes[2].imshow(results_60[0].plot())
    axes[2].set_title('Previsão - Modelo 60 épocas', fontsize=12)
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()
    print("Detalhes da previsão - Modelo 30 épocas:")
    for box in results_30[0].boxes:
        classe = classes[int(box.cls)]
        confianca = float(box.conf)
        print(f"  - Classe: {classe}, Confiança: {confianca:.4f}")
    print("\nDetalhes da previsão - Modelo 60 épocas:")
    for box in results_60[0].boxes:
        classe = classes[int(box.cls)]
        confianca = float(box.conf)
        print(f"  - Classe: {classe}, Confiança: {confianca:.4f}")

val_imgs = glob.glob(os.path.join(val_dir, '*.jpg')) + glob.glob(os.path.join(val_dir, '*.png'))
if val_imgs:
    random.shuffle(val_imgs)
    for img_path in val_imgs[:3]:
        print(f"\nAnalisando: {os.path.basename(img_path)}")
        show_predictions_comparison(img_path)
else:
    print("Nenhuma imagem de validação encontrada!")

## 7. Teste do Modelo em Imagens Separadas (Fase 4)

Utilize as imagens de teste para avaliar o modelo treinado. Salve os resultados visuais e calcule métricas específicas para o conjunto de teste.

In [ ]:
test_imgs = glob.glob(os.path.join(test_dir, '*.jpg'))
os.makedirs(os.path.join(results_dir, 'test_preds'), exist_ok=True)
for img_path in test_imgs:
    results = model(img_path, weights=os.path.join(models_dir, 'model_60epochs.pt'))
    # Salvar imagem com bounding boxes
    results[0].save(filename=os.path.join(results_dir, 'test_preds', os.path.basename(img_path)))

### Cálculo de Métricas no Conjunto de Teste

Avalie precisão, recall, mAP e identifique falsos positivos/negativos.

In [ ]:
# Exemplo de avaliação no conjunto de teste
test_metrics = model.val(data=data_yaml, batch=batch_size, imgsz=imgsz, split='test', weights=os.path.join(models_dir, 'model_60epochs.pt'))
print('Métricas no conjunto de teste:', test_metrics)

### Análise de Casos de Sucesso e Limitações

Documente exemplos de detecções corretas, falsos positivos e falsos negativos. Discuta limitações observadas e possíveis melhorias.

> **TO-DO:** Insira imagens de casos de sucesso, falsos positivos/negativos e análise crítica dos resultados.

## 8. Conclusões e Próximos Passos

- Resuma os principais resultados obtidos
- Discuta pontos fortes e limitações do modelo
- Sugira melhorias e próximos experimentos


> **TO-DO:** Preencher com conclusões, gráficos e sugestões para fases futuras (YOLO tradicional, CNN do zero, análise comparativa etc.).